In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('ggplot')
plt.rcParams['grid.color'] = 'white'
from scipy.optimize import curve_fit
import scipy.integrate as spi
import pandas as pd
import glob
import seaborn as sns
from kin_functions import *

# Different simulations

In [ ]:
# substrate info
# #               name,          NoC,    NoE
info_glu = [    'glucose',       6,      24]
info_ac  = [    'acetate',       2,      8 ]

# total electrons in feed
e_tot_in = 0.5     # emol/L 

# kinetic parameters
K_glu = 10*10**(-6)    # M = 0.01 mM     
K_ac = K_glu

## Rewrite acetate specialist Ks

In [ ]:
Y_ac_an_tot = -0.5234355
Y_ac_met_tot = -1.015459723
Y_ac_cat_tot = Y_ac_met_tot - Y_ac_an_tot
mu_max_ac = 0.49780853
q_s_cat_max_ac = -2 / info_ac[-1]        # mol S/Cmol X/h
q_s_max_ac = mu_max_ac * Y_ac_met_tot          # mol S/Cmol X/h

Kc_fit_eD2, Ke_fit_eD2, _ = fit_kinetics(q_s_max_ac, K_ac, Y_ac_an_tot, Y_ac_cat_tot, q_s_cat_max_ac, mu_max_ac, info_ac[0], info_ac[-1])

## $f_{eD1}$ = $f_{feed}$

### µ > µ$_{max,acetate}$ and f $_{eD1}$ < opt

In [ ]:
# specify the substrate fraction in metabolism, how much of the total electrons come from eD1 
f_eD1_i = 0.67     # emol eD1/emol tot

# specify the substrate fraction in the feed
f_feed = f_eD1_i

# operational parameters: dilution rate (= growth rate), hydraulic retention time (HRT) and total substrate fed
D        = 0.6      # 1/h
HRT      = 1/D      # h

# run simulation
sol_1, kin_figs_1 = simulate_competition("info_summary_glu_ac.xlsx", info_glu, info_ac, K_glu, K_ac, f_eD1_i, f_feed, D, e_tot_in, 100)



### µ > µ$_{max,acetate}$ and f $_{eD1}$ = opt

In [ ]:
# specify the substrate fraction in metabolism, how much of the total electrons come from eD1 
f_eD1_i = 0.75    # emol eD1/emol tot

# specify the substrate fraction in the feed
f_feed = f_eD1_i

# operational parameters: dilution rate (= growth rate), hydraulic retention time (HRT) and total substrate fed
D        = 0.6      # 1/h
HRT      = 1/D      # h

# run simulation
sol_2, kin_figs_2 = simulate_competition("info_summary_glu_ac.xlsx",info_glu, info_ac, K_glu, K_ac, f_eD1_i, f_feed, D, e_tot_in, 100)

#### The kinetic fit

In [ ]:
for fig in kin_figs_2:
    display(fig)

### µ > µ$_{max,acetate}$ and f $_{eD1}$ > opt 

In [ ]:
# specify the substrate fraction in metabolism, how much of the total electrons come from eD1 
f_eD1_i = 0.8572     # emol eD1/emol tot

# specify the substrate fraction in the feed
f_feed = f_eD1_i

# operational parameters: dilution rate (= growth rate), hydraulic retention time (HRT) and total substrate fed
D        = 0.6      # 1/h
HRT      = 1/D      # h

# run simulation
sol_3, kin_figs_3 = simulate_competition("info_summary_glu_ac.xlsx",info_glu, info_ac, K_glu, K_ac, f_eD1_i, f_feed, D, e_tot_in, 100)


### µ < µ$_{max,acetate}$ and f $_{eD1}$ < opt

In [ ]:
# specify the substrate fraction in metabolism, how much of the total electrons come from eD1 
f_eD1_i = 0.67     # emol eD1/emol tot

# specify the substrate fraction in the feed
f_feed = f_eD1_i

# operational parameters: dilution rate (= growth rate), hydraulic retention time (HRT) and total substrate fed
D        = 0.1      # 1/h
HRT      = 1/D      # h

# run simulation
sol_4, kin_figs_4 = simulate_competition("info_summary_glu_ac.xlsx",info_glu, info_ac, K_glu, K_ac, f_eD1_i, f_feed, D, e_tot_in, 100)


### µ < µ$_{max,acetate}$ and f $_{eD1}$ = opt

In [ ]:
# specify the substrate fraction in metabolism, how much of the total electrons come from eD1 
f_eD1_i = 0.75     # emol eD1/emol tot

# specify the substrate fraction in the feed
f_feed = f_eD1_i

# operational parameters: dilution rate (= growth rate), hydraulic retention time (HRT) and total substrate fed
D        = 0.1      # 1/h
HRT      = 1/D      # h

# run simulation
sol_5, kin_figs_5 = simulate_competition("info_summary_glu_ac.xlsx",info_glu, info_ac, K_glu, K_ac, f_eD1_i, f_feed, D, e_tot_in, 100)

### µ < µ$_{max,acetate}$ and f $_{eD1}$ > opt

In [ ]:
# specify the substrate fraction in metabolism, how much of the total electrons come from eD1 
f_eD1_i = 0.8572     # emol eD1/emol tot

# specify the substrate fraction in the feed
f_feed = f_eD1_i

# operational parameters: dilution rate (= growth rate), hydraulic retention time (HRT) and total substrate fed
D        = 0.1      # 1/h
HRT      = 1/D      # h

# run simulation
sol_6, kin_figs_6 = simulate_competition("info_summary_glu_ac.xlsx",info_glu, info_ac, K_glu, K_ac, f_eD1_i, f_feed, D, e_tot_in, 100)

## VaryingK1/K2

In [ ]:
# Parameter grid
f_eD1_vals = np.arange(0.1, 1.0, 0.05)   # 0.0, 0.1, …, 1.0
rK1_K2_vals = np.logspace(-2, 2, 5)               # biologically relevant range, considering that K_glu is the reference value


In [ ]:
survival_matrix_spec_var_lowD = np.zeros((len(f_eD1_vals), len(rK1_K2_vals)), dtype=int)

# increase K_glu one order of magnitude, to avoid problems of negative electron/carbon affinities
K_glu = K_glu * 10 

D = 0.1
HRT_default = 30   # long enough for quasi-steady state

for i_f, f_eD1 in enumerate(f_eD1_vals):
    for i_k, rK1_K2_i in enumerate(rK1_K2_vals):
        print(f'{f_eD1} and Ks ratio = {rK1_K2_i}')

        K_ac = K_glu / rK1_K2_i

        sol, _ = simulate_competition("info_summary_glu_ac.xlsx",
            info_glu, info_ac,
            K_glu, K_ac,
            f_eD1, f_eD1,
            D,
            e_tot_in,
            HRT_default,
            dt=0.01)#, comp_plots=False)

        # species concentrations
        _, _, _, _, c_X3 = sol.y

        # Only track species 3 survival
        survival_matrix_spec_var_lowD[i_f, i_k] = int(survives_species3(c_X3))


In [ ]:
survival_matrix_spec_var_highD = np.zeros((len(f_eD1_vals), len(rK1_K2_vals)), dtype=int)

D = 0.6
HRT_default = 30   # long enough for quasi-steady state

for i_f, f_eD1 in enumerate(f_eD1_vals):
    for i_k, rK1_K2_i in enumerate(rK1_K2_vals):
        print(f'{f_eD1} and Ks ratio = {rK1_K2_i}')

        K_ac = K_glu / rK1_K2_i

        sol, _ = simulate_competition("info_summary_glu_ac.xlsx",
            info_glu, info_ac,
            K_glu, K_ac,
            f_eD1, f_eD1,
            D,
            e_tot_in,
            HRT_default,
            dt=0.01)#, comp_plots=False)

        # species concentrations
        _, _, _, _, c_X3 = sol.y

        # Only track species 3 survival
        survival_matrix_spec_var_highD[i_f, i_k] = int(survives_species3(c_X3))


## When generalist is fitted to a _different_ glucose specialist

In [ ]:
survival_matrix_glu_gen_lowD = np.zeros((len(f_eD1_vals), len(rK1_K2_vals)), dtype=int)

D = 0.1
HRT_default = 30   # long enough for quasi-steady state

for i_f, f_eD1 in enumerate(f_eD1_vals):
    for i_k, rK1_K2_i in enumerate(rK1_K2_vals):
        print(f'{f_eD1} and Ks ratio = {rK1_K2_i}')

        K_ac = K_glu 
        K_glu_gen_i = K_glu / rK1_K2_i

        sol, _ = simulate_competition("info_summary_glu_ac.xlsx",
            info_glu, info_ac,
            K_glu, K_ac,
            f_eD1, f_eD1,
            D,
            e_tot_in,
            HRT_default,
            dt=0.01, K_eD1_gen = K_glu_gen_i)#, comp_plots=False)

        # species concentrations
        _, _, _, _, c_X3 = sol.y

        # Only track species 3 survival
        survival_matrix_glu_gen_lowD[i_f, i_k] = int(survives_species3(c_X3))


In [ ]:
survival_matrix_glu_gen_highD = np.zeros((len(f_eD1_vals), len(rK1_K2_vals)), dtype=int)

D = 0.6
HRT_default = 30   # long enough for quasi-steady state

for i_f, f_eD1 in enumerate(f_eD1_vals):
    for i_k, rK1_K2_i in enumerate(rK1_K2_vals):
        print(f'{f_eD1} and Ks ratio = {rK1_K2_i}')

        K_ac = K_glu 
        K_glu_gen_i = K_glu / rK1_K2_i

        sol, _ = simulate_competition("info_summary_glu_ac.xlsx",
            info_glu, info_ac,
            K_glu, K_ac,
            f_eD1, f_eD1,
            D,
            e_tot_in,
            HRT_default,
            dt=0.01, K_eD1_gen = K_glu_gen_i)#, comp_plots=False)

        # species concentrations
        _, _, _, _, c_X3 = sol.y

        # Only track species 3 survival
        survival_matrix_glu_gen_highD[i_f, i_k] = int(survives_species3(c_X3))

## Save the outcomes

In [ ]:
np.save("glu_ac_high_D_var_gen.npy", survival_matrix_glu_gen_highD)
np.save("glu_ac_high_D_var_spec.npy", survival_matrix_spec_var_highD)
np.save("glu_ac_low_D_var_gen.npy", survival_matrix_glu_gen_lowD)
np.save("glu_ac_low_D_var_spec.npy", survival_matrix_spec_var_lowD)